In [ ]:
import cv2
import os
import numpy as np

# continue going one folder up as long as folder name is not 'move_deve'
while os.path.basename(os.getcwd()) != 'photostim_deve':
    os.chdir('..')

move_deve_path = os.getcwd()

In [ ]:
experimenter = 'jm'
data_raw_dir = 'data_raw/'
data_vid_dir = 'data_vid/'

fps = 30  # output fps for the generated videos

start_frame = 17940     # first frame index (within each session) to include
end_frame = 17940 + 800      # last frame index (exclusive) to include
n_sessions = 6        # number of sessions to use, first 6 '_s' dirs per mouse
grid_rows, grid_cols = 2, 3  # layout for the grid video (rows x cols == n_sessions)

out_format = 'avi'   # 'avi' or 'mp4'
fourcc_map = {'avi': 'MJPG', 'mp4': 'mp4v'}
fourcc = cv2.VideoWriter_fourcc(*fourcc_map[out_format])

In [ ]:
# get all mice (subject dirs)
all_subject_dirs = ['jm064', 'jm065', 'jm067', 'jm073', 'jm074', 'jm075', 'jm083', 'jm084', 'jm089', 'jm090', 'jm091']

In [ ]:
def get_subject_session_images(subject_dir, n_sessions):
    """Return (session_dirs, session_images) for the first n_sessions '_s' dirs of a mouse.

    session_images[i] is the sorted list of tiff paths for that session's camera folder
    (empty list if the session or its camera folder doesn't exist).
    Reads directly from data_raw, not from any pre-compressed videos.
    """
    session_root = data_raw_dir + experimenter + '/' + subject_dir
    all_session_dirs = sorted(os.listdir(session_root))
    all_session_dirs = [x for x in all_session_dirs if x[-2:] == '_s'][:n_sessions]

    session_images = []
    for session_dir in all_session_dirs:
        camera_dir = session_root + '/' + session_dir + '/camera'
        if os.path.isdir(camera_dir):
            images = sorted([os.path.join(camera_dir, f) for f in os.listdir(camera_dir) if f.endswith('.tiff')])
        else:
            images = []
        session_images.append(images)

    return all_session_dirs, session_images

In [ ]:
# --- grid video: sessions shown side by side, synchronized frame-by-frame ---
# saved at data_vid/<experimenter>/<subject_dir>/crossday/grid.<out_format>

for subject_dir in all_subject_dirs:
    all_session_dirs, session_images = get_subject_session_images(subject_dir, n_sessions)

    # pad up to n_sessions (in case fewer than n_sessions session dirs exist)
    while len(session_images) < n_sessions:
        all_session_dirs.append(None)
        session_images.append([])

    if all(len(imgs) == 0 for imgs in session_images):
        print(f"No available sessions with camera data for {subject_dir}, skipping grid...")
        continue

    # tile size taken from the first frame available at start_frame
    tile_w, tile_h = None, None
    for imgs in session_images:
        if len(imgs) > start_frame:
            sample = cv2.imread(imgs[start_frame])
            tile_h, tile_w = sample.shape[:2]
            break
    if tile_w is None:
        print(f"No frames available at start_frame={start_frame} for {subject_dir}, skipping grid...")
        continue

    out_dir = data_vid_dir + experimenter + '/' + subject_dir + '/crossday/'
    os.makedirs(out_dir, exist_ok=True)
    out_path = out_dir + f'grid_evoked.{out_format}'

    video_writer = cv2.VideoWriter(out_path, fourcc, fps, (tile_w * grid_cols, tile_h * grid_rows))

    n_frames = end_frame - start_frame
    for frame_idx in range(start_frame, end_frame):
        tiles = []
        for imgs in session_images:
            if frame_idx < len(imgs):
                tile = cv2.imread(imgs[frame_idx])
                if tile.shape[:2] != (tile_h, tile_w):
                    tile = cv2.resize(tile, (tile_w, tile_h))
            else:
                # session missing or ran out of frames -> black tile
                tile = np.zeros((tile_h, tile_w, 3), dtype=np.uint8)
            tiles.append(tile)

        rows = [np.hstack(tiles[r * grid_cols:(r + 1) * grid_cols]) for r in range(grid_rows)]
        grid_frame = np.vstack(rows)
        video_writer.write(grid_frame)

        if (frame_idx - start_frame) % 500 == 0:
            print(f"{subject_dir}: processed {frame_idx - start_frame} / {n_frames} frames")

    video_writer.release()
    print(f"Saved grid video for {subject_dir} at {out_path}")

In [ ]:
# --- concat video: sessions played back to back, in day order (no stacking) ---
# saved at data_vid/<experimenter>/<subject_dir>/crossday/concat.<out_format>

for subject_dir in all_subject_dirs:
    all_session_dirs, session_images = get_subject_session_images(subject_dir, n_sessions)

    # keep only sessions that actually have frames in the requested range
    available = [(d, imgs) for d, imgs in zip(all_session_dirs, session_images) if len(imgs) > start_frame]

    if not available:
        print(f"No available sessions with camera data for {subject_dir}, skipping concat...")
        continue

    # frame size taken from the first available session
    sample = cv2.imread(available[0][1][start_frame])
    tile_h, tile_w = sample.shape[:2]

    out_dir = data_vid_dir + experimenter + '/' + subject_dir + '/crossday/'
    os.makedirs(out_dir, exist_ok=True)
    out_path = out_dir + f'concat.{out_format}'

    video_writer = cv2.VideoWriter(out_path, fourcc, fps, (tile_w, tile_h))

    for session_dir, images in available:
        frame_end = min(end_frame, len(images))
        for frame_idx in range(start_frame, frame_end):
            frame = cv2.imread(images[frame_idx])
            if frame.shape[:2] != (tile_h, tile_w):
                frame = cv2.resize(frame, (tile_w, tile_h))
            video_writer.write(frame)
        print(f"{subject_dir}: appended {session_dir} ({frame_end - start_frame} frames)")

    video_writer.release()
    print(f"Saved concatenated cross-day video for {subject_dir} at {out_path}")